In [6]:
from datasets import load_dataset
from pprint import pprint

In [7]:
dataset = load_dataset("tner/bc5cdr")
print(dataset["train"][0])

{'tokens': ['Naloxone', 'reverses', 'the', 'antihypertensive', 'effect', 'of', 'clonidine', '.'], 'tags': [1, 0, 0, 0, 0, 0, 1, 0]}


In [18]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")
tokens = tokenizer(
    dataset["train"][0]["tokens"],
    is_split_into_words=True,
    truncation=True,
    padding="max_length",
    max_length=64,
    return_tensors="pt"
    )


print(tokens)

{'input_ids': tensor([[  101, 11896,  2858, 21501,  1162,  7936,  1116,  1103,  2848,  7889,
         17786,  5026,  2109,  2629,  1104,   172,  4934,  2386,  2042,   119,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [19]:
print(tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))

['[CLS]', 'Na', '##lo', '##xon', '##e', 'reverse', '##s', 'the', 'anti', '##hy', '##pert', '##ens', '##ive', 'effect', 'of', 'c', '##lon', '##id', '##ine', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [20]:
word_ids = tokens.word_ids()
print(word_ids)

[None, 0, 0, 0, 0, 1, 1, 2, 3, 3, 3, 3, 3, 4, 5, 6, 6, 6, 6, 7, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]


In [21]:
sample = dataset["train"][0]

tokens = tokenizer(
    sample["tokens"],
    is_split_into_words=True
)

word_ids = tokens.word_ids()

aligned_labels = []

for word_id in word_ids:
    if word_id is None:
        aligned_labels.append(-100)
    else:
        aligned_labels.append(sample["tags"][word_id])

print(aligned_labels)

[-100, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, -100]


In [25]:
print(tokens.word_ids())
print(aligned_labels)

[None, 0, 0, 0, 0, 1, 1, 2, 3, 3, 3, 3, 3, 4, 5, 6, 6, 6, 6, 7, None]
[-100, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, -100]


In [26]:
for token, word_id, label in zip(tokens, word_ids, aligned_labels):
    print(f"{token:15} {str(word_id):5} {label}")

input_ids       None  -100
token_type_ids  0     1
attention_mask  0     1


In [32]:
aligned_labels = []

label_map = {
    1: 4,
    2: 3
}

for sample in dataset["train"]:
    tokens = sample["tokens"]
    tags = sample["tags"]
    
    tokenized = tokenizer(
        tokens,
        is_split_into_words=True
    )

    word_ids=tokenized.word_ids()

    labels = []
    previous_word_id = None

    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
        
        elif word_id != previous_word_id:
            labels.append(tags[word_id])

        else:
            label = tags[word_id]

            if label in label_map:
                label = label_map[label]
            
            labels.append(label)

        previous_word_id = word_id
    aligned_labels.append(labels)

print(aligned_labels)

[[-100, 1, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 4, 4, 4, 0, -100], [-100, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 4, 4, 0, 0, 0, -100], [-100, 0, 0, 0, 0, 0, 0, 0, -100], [-100, 0, 2, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 1, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 1, 4, 4, 4, 0, -100], [-100, 1, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100], [-100, 0, 0, 0, 0, 0, 0, 0, 2, 3, 3, 0, 1, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 1, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 4, 4, 4, 0, 0, 0, 0, 0, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -100], [-100, 0, 1, 4, 4, 4, 0, 1, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 4, 4, 4, 0, 1, 4, 4,

In [33]:
tokens = tokenizer.convert_ids_to_tokens(tokenized["input_ids"])

for token, word_id, label in zip(tokens, word_ids, labels):
    print(f"{token:15} {str(word_id):5} {label}")

[CLS]           None  -100
However         0     0
,               1     0
a               2     1
##po            2     4
##mor           2     4
##phine         2     4
-               3     0
induced         4     0
reduction       5     0
##s             5     0
in              6     0
s               7     0
##tri           7     0
##ata           7     0
##l             7     0
do              8     1
##pa            8     4
##mine          8     4
[SEP]           None  -100


In [34]:
import torch

print(torch.cuda.is_available())

False
